# 品类关联（购物篮）

没有 FP-Growth 依赖，直接用 SQL 自关联算品类共现，再用 pandas 算支持度、置信度和提升度。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
pairs = query("""
SELECT a.category AS c1, b.category AS c2,
       COUNT(*) AS pair_orders
FROM (SELECT DISTINCT order_id, category FROM dws_order_wide) a
JOIN (SELECT DISTINCT order_id, category FROM dws_order_wide) b
  ON a.order_id = b.order_id AND a.category < b.category
GROUP BY a.category, b.category
HAVING pair_orders >= 100
ORDER BY pair_orders DESC
""")

In [3]:
cat = query("""
SELECT category, COUNT(DISTINCT order_id) AS orders
FROM dws_order_wide
GROUP BY category
""")
cat["orders"] = cat["orders"].astype(int)
cat = cat.set_index("category")["orders"]

total_orders = int(query("SELECT COUNT(DISTINCT order_id) AS n FROM dws_order_wide").n[0])

pairs["pair_orders"] = pairs["pair_orders"].astype(float)
pairs["c1_orders"] = pairs.c1.map(cat).astype(float)
pairs["c2_orders"] = pairs.c2.map(cat).astype(float)

pairs["support"] = (pairs.pair_orders / total_orders).round(4)
pairs["conf_c1_to_c2"] = (pairs.pair_orders / pairs.c1_orders).round(3)
pairs["lift"] = (pairs.pair_orders * total_orders / (pairs.c1_orders * pairs.c2_orders)).round(2)

cols = ["c1", "c2", "pair_orders", "support", "conf_c1_to_c2", "lift"]
pairs[pairs.lift > 1].sort_values("lift", ascending=False)[cols].head(10)


,c1,c2,pair_orders,support,conf_c1_to_c2,lift


提升度大于 1 才是真关联。只看共现次数会把家居这类人人都买的刚需品类排到前面，加上 lift 之后才能看出真正的组合。